# 07_sample_source_docs

MongoDB の JaLC メタデータから評価用の元データをサンプリングし、固定ファイルへ保存する。

In [ ]:
# Notebook から src 配下の自作モジュールを import できるようにする設定
from pathlib import Path
import sys

project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
# 設定値と評価用元データサンプリング関数を読み込む
from pprint import pprint

from tanigawa_shoshi.config import SAMPLED_SOURCE_DOCS_PATH
from tanigawa_shoshi.evaluation_data import (
    load_source_docs,
    sample_source_docs,
    save_source_docs,
)


In [ ]:
# サンプリング条件を指定する
sample_size = 50
seed = 42
max_attempts = sample_size * 100

print(f"sample_size: {sample_size}")
print(f"seed: {seed}")
print(f"max_attempts: {max_attempts}")
print(f"output_path: {SAMPLED_SOURCE_DOCS_PATH}")


In [ ]:
# MongoDB から評価用元メタデータをサンプリングする
sample_result = sample_source_docs(
    sample_size=sample_size,
    seed=seed,
    max_attempts=max_attempts,
)

print("sampling stats")
pprint(sample_result["stats"])
print()
print("id bounds")
print(sample_result["min_object_id"])
print(sample_result["max_object_id"])
print()
print(f"sampled docs: {len(sample_result['docs'])}")

# attempt_count: サンプリングを試行した回数
# sampled_count: 最終的に採用できた文書数
# duplicate_id_count: 取得した文書の_idがすでに採用済みだった回数
# duplicate_doi_count: 取得した文書のdoiがすでに採用済みだった回数
# missing_candidate_count: サンプリング条件に合う文書が見つからなかった回数
# rejected_missing_doi_count: doiがない文書を除外した回数
# rejected_missing_required_fields_count: 必須フィールドがない文書を除外した回数

In [ ]:
# サンプリング結果を固定ファイルへ保存する
saved_path = save_source_docs(sample_result["docs"])
print(saved_path)


In [ ]:
# 保存した元メタデータを再読込し、先頭数件を確認する
source_docs = load_source_docs()

print(f"loaded docs: {len(source_docs)}")
print()

for index, doc in enumerate(source_docs[:3], start=1):
    print("=" * 60)
    print(f"index: {index}")
    print(f"_id: {doc.get('_id')}")
    print(f"doi: {doc.get('doi')}")
    print(f"publication_date: {doc.get('publication_date')}")
    print(f"volume: {doc.get('volume')}")
    print(f"issue: {doc.get('issue')}")
    print(f"first_page: {doc.get('first_page')}")
    print(f"last_page: {doc.get('last_page')}")
    print(f"creator_count: {len(doc.get('creator_list') or [])}")
    print(f"title_count: {len(doc.get('title_list') or [])}")
    print(f"journal_count: {len(doc.get('journal_title_name_list') or [])}")
